In [4]:
import pandas as pd
from utils.constants import *
import shutil
#import kagglehub
"""
pip install kagglehub[pandas-datasets] to download dataset
 

os.makedirs('gtrsb_dataset', exist_ok=True)
dataset_path = kagglehub.dataset_download("meowmeowmeowmeowmeow/gtsrb-german-traffic-sign")
print(f"Dataset downloaded to: {dataset_path}")

# move the dataset to current project directory
target_path = './gtrsb_dataset'
if os.path.exists(target_path):
    shutil.move(dataset_path, target_path)


pd.read_csv(os.path.join(DF_PATH, 'Test.csv'))
"""

'\npip install kagglehub[pandas-datasets] to download dataset\n \n\nos.makedirs(\'gtrsb_dataset\', exist_ok=True)\ndataset_path = kagglehub.dataset_download("meowmeowmeowmeowmeow/gtsrb-german-traffic-sign")\nprint(f"Dataset downloaded to: {dataset_path}")\n\n# move the dataset to current project directory\ntarget_path = \'./gtrsb_dataset\'\nif os.path.exists(target_path):\n    shutil.move(dataset_path, target_path)\n\n\npd.read_csv(os.path.join(DF_PATH, \'Test.csv\'))\n'

In [26]:
train_df = pd.read_csv(os.path.join(DF_PATH, 'Train.csv'))
test_df = pd.read_csv(os.path.join(DF_PATH, 'Test.csv'))
meta_df = pd.read_csv(os.path.join(DF_PATH, 'Meta.csv'))

In [27]:
for df in [train_df, test_df]:
      df["roi_w"] = (df["Roi.X2"] - df["Roi.X1"]).clip(lower=1)
      df["roi_h"] = (df["Roi.Y2"] - df["Roi.Y1"]).clip(lower=1)
      df["img_area"] = df["Width"] * df["Height"]
      df["roi_area"] = df["roi_w"] * df["roi_h"]
      df["roi_cover"] = (df["roi_area"] / df["img_area"]).clip(upper=1.0)

train_df.head(3)

,Width,Height,Roi.X1,Roi.Y1,Roi.X2,Roi.Y2,ClassId,Path,roi_w,roi_h,img_area,roi_area,roi_cover
0,27,26,5,5,22,20,20,Train/20/00020_00000_00000.png,17,15,702,255,0.363248
1,28,27,5,6,23,22,20,Train/20/00020_00000_00001.png,18,16,756,288,0.380952
2,29,26,6,5,24,21,20,Train/20/00020_00000_00002.png,18,16,754,288,0.381963


In [28]:
from sklearn.svm import SVC
class_column = meta_df['ClassId'].to_list()

test_df = test_df.merge(
    meta_df[['ClassId', 'ColorId', 'ShapeId', 'SignId']],
    how='left',      # keep all rows in test_df
    on='ClassId'     # match by ClassId
)
train_df = train_df.merge(
    meta_df[['ClassId', 'ColorId', 'ShapeId', 'SignId']],
    how='left',      # keep all rows in test_df
    on='ClassId'     # match by ClassId
)


#test_df.rename(columns={'ColorId': 'Color'}, inplace=True)
train_df

,Width,Height,Roi.X1,Roi.Y1,Roi.X2,Roi.Y2,ClassId,Path,roi_w,roi_h,img_area,roi_area,roi_cover,ColorId,ShapeId,SignId
0,27,26,5,5,22,20,20,Train/20/00020_00000_00000.png,17,15,702,255,0.363248,0,0,1.1
1,28,27,5,6,23,22,20,Train/20/00020_00000_00001.png,18,16,756,288,0.380952,0,0,1.1
2,29,26,6,5,24,21,20,Train/20/00020_00000_00002.png,18,16,754,288,0.381963,0,0,1.1
3,28,27,5,6,23,22,20,Train/20/00020_00000_00003.png,18,16,756,288,0.380952,0,0,1.1
4,28,26,5,5,23,21,20,Train/20/00020_00000_00004.png,18,16,728,288,0.395604,0,0,1.1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39204,52,56,5,6,47,51,42,Train/42/00042_00007_00025.png,42,45,2912,1890,0.649038,3,1,3.28
39205,56,58,5,5,51,53,42,Train/42/00042_00007_00026.png,46,48,3248,2208,0.679803,3,1,3.28
39206,58,62,5,6,53,57,42,Train/42/00042_00007_00027.png,48,51,3596,2448,0.680756,3,1,3.28
39207,63,69,5,7,58,63,42,Train/42/00042_00007_00028.png,53,56,4347,2968,0.682770,3,1,3.28


In [29]:
test_df

,Width,Height,Roi.X1,Roi.Y1,Roi.X2,Roi.Y2,ClassId,Path,roi_w,roi_h,img_area,roi_area,roi_cover,ColorId,ShapeId,SignId
0,53,54,6,5,48,49,16,Test/00000.png,42,44,2862,1848,0.645702,0,1,3.3
1,42,45,5,5,36,40,1,Test/00001.png,31,35,1890,1085,0.574074,0,1,3.29
2,48,52,6,6,43,47,38,Test/00002.png,37,41,2496,1517,0.607772,1,1,4.7
3,27,29,5,5,22,24,33,Test/00003.png,17,19,783,323,0.412516,1,1,4.2
4,60,57,5,5,55,52,11,Test/00004.png,50,47,3420,2350,0.687135,0,0,1.22
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12625,42,41,5,6,37,36,12,Test/12625.png,32,30,1722,960,0.557491,2,2,2.3
12626,50,51,6,5,45,46,33,Test/12626.png,39,41,2550,1599,0.627059,1,1,4.2
12627,29,29,6,6,24,24,6,Test/12627.png,18,18,841,324,0.385256,3,1,3.3
12628,48,49,5,6,43,44,7,Test/12628.png,38,38,2352,1444,0.613946,0,1,3.29


array(['3.3', '3.29', '4.7', '4.2', '1.22', '1.39', '2.3', '1.37', '4.1',
       '1.13', '3.25', '1.3.2', '1.1', '1.32', '2.1', '3.27', '3.21',
       '4.3', nan, '1.5.2', '3.1', '1.24', '1.33', '2.2', '3.42', '1.34',
       '4.4', '3.26', '1.36', '1.2', '4.8', '3.28', '4.5'], dtype=object)

In [40]:
from sklearn.neighbors import KNeighborsClassifier

columns = [['ColorId', 'ShapeId']]
train = train_df[['ColorId', 'ShapeId']]
test = test_df[['ColorId', 'ShapeId']]
#test['SignId'] = test['SignId'].astype('category').cat.codes
#train['SignId'] = train['SignId'].astype('category').cat.codes

model = KNeighborsClassifier(n_neighbors= 5)
model.fit(train, train_df['ClassId'])
model.predict(test)


array([ 0,  0, 33, ...,  6,  0,  0], dtype=int64)

In [42]:
from sklearn.metrics import classification_report,\
    confusion_matrix

model = KNeighborsClassifier(n_neighbors= 10)
model.fit(train, train_df['ClassId'])
pred = model.predict(test)
#print(confusion_matrix(test_df['ClassId'], pred))
print(classification_report(test_df['ClassId'], pred))

              precision    recall  f1-score   support

           0       0.01      1.00      0.02        60
           1       0.00      0.00      0.00       720
           2       0.00      0.00      0.00       750
           3       0.00      0.00      0.00       450
           4       0.00      0.00      0.00       660
           5       0.00      0.00      0.00       630
           6       0.42      1.00      0.59       150
           7       0.00      0.00      0.00       450
           8       0.00      0.00      0.00       450
           9       0.00      0.00      0.00       480
          10       0.00      0.00      0.00       660
          11       0.00      0.00      0.00       420
          12       1.00      1.00      1.00       690
          13       1.00      1.00      1.00       720
          14       1.00      1.00      1.00       270
          15       0.00      0.00      0.00       210
          16       0.00      0.00      0.00       150
          17       0.00    

c:\Users\aless\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\aless\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\aless\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [46]:
meta_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 43 entries, 0 to 42
Data columns (total 5 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   Path     43 non-null     object
 1   ClassId  43 non-null     int64 
 2   ShapeId  43 non-null     int64 
 3   ColorId  43 non-null     int64 
 4   SignId   42 non-null     object
dtypes: int64(3), object(2)
memory usage: 1.8+ KB
